# EMRGY Valuation Architecture Demo

This notebook shows the valuation engine as a simple pipeline:

```text
Site + annual resource assumptions
  -> physical assets
  -> shared assets
  -> standardized outputs
  -> revenue parameters
  -> project cash flows
  -> NPV / IRR / LCOE / payback
```

The important modeling boundary is that assets produce physical outputs. Revenue models assign dollar values to those outputs.

## 1. Setup

In [1]:
from pathlib import Path
import html
import sys

for src_path in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if src_path.exists():
        sys.path.insert(0, str(src_path))
        break

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None

    def display(value):
        print(value)

from emrgy_valuation import (
    AssetPortfolio,
    Cabling,
    Canal,
    CanalLining,
    CapacityPaymentRevenue,
    EmrgyTurbineArray,
    FinanceAssumptions,
    FinanceModel,
    FixedPriceEnergyRevenue,
    FloatingPVArray,
    ResourceProfile,
    Site,
    ValuationEngine,
    WaterSavingsRevenue,
)


def usd(value, digits=0):
    return f"${value:,.{digits}f}"


def number(value, digits=1):
    return f"{value:,.{digits}f}"


def percent(value):
    return "n/a" if value is None else f"{value:.1%}"


def table(headers, rows):
    if HTML is None:
        print("\t".join(str(header) for header in headers))
        for row in rows:
            print("\t".join(str(cell) for cell in row))
        return

    header_html = "".join(f"<th>{html.escape(str(header))}</th>" for header in headers)
    body_html = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(cell))}</td>" for cell in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"""
    <style>
      table.emrgy-demo {{ border-collapse: collapse; margin: 0.75rem 0 1.25rem 0; min-width: 720px; }}
      table.emrgy-demo th, table.emrgy-demo td {{ border: 1px solid #ddd; padding: 7px 10px; text-align: left; }}
      table.emrgy-demo th {{ background: #f6f8fa; }}
      table.emrgy-demo td:nth-child(n+2) {{ white-space: nowrap; }}
    </style>
    <table class="emrgy-demo"><thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody></table>
    """))

print("Ready.")

Ready.


## 2. Define The Site And Annual Resource

The site captures the canal context. The resource profile is annual: one representative water velocity, depth, solar capacity factor, availability, and hour count.

In [2]:
site = Site(
    name="Demo canal hybrid",
    canal=Canal(bottom_width_m=5.0, usable_length_m=300.0, access_road=True),
    utility_territory="Example utility",
    tax_jurisdiction="US",
)

resources = ResourceProfile(
    annual_hours=8760,
    water_velocity_m_s=1.38,
    water_depth_m=1.10,
    solar_capacity_factor=0.218,
    availability=0.94,
)

table(
    ["Input", "Value"],
    [
        ("Site", site.name),
        ("Canal bottom width", f"{site.canal.bottom_width_m:.1f} m"),
        ("Usable length", f"{site.canal.usable_length_m:.0f} m"),
        ("Water surface area", f"{site.canal.water_surface_area_m2:.0f} m2"),
        ("Annual hours", f"{resources.annual_hours:,.0f}"),
        ("Annual water velocity", f"{resources.water_velocity_m_s:.2f} m/s"),
        ("Annual water depth", f"{resources.water_depth_m:.2f} m"),
        ("Solar capacity factor", f"{resources.solar_capacity_factor:.1%}"),
        ("Availability", f"{resources.availability:.1%}"),
    ],
)

Input,Value
Site,Demo canal hybrid
Canal bottom width,5.0 m
Usable length,300 m
Water surface area,1500 m2
Annual hours,"8,760"
Annual water velocity,1.38 m/s
Annual water depth,1.10 m
Solar capacity factor,21.8%
Availability,94.0%


## 3. Define Physical Assets

Physical assets produce physical outputs and their own costs. They do not decide how those outputs get paid.

In [3]:
physical_assets = [
    EmrgyTurbineArray(name="EMRGY turbines", turbine_count=2),
    FloatingPVArray(name="Floating PV", dc_capacity_kw=75),
    CanalLining(
        name="Canal lining",
        lined_area_m2=500,
        capex_per_m2_usd=45,
        avoided_water_loss_m3_per_year=8000,
    ),
]

table(
    ["Physical asset", "Key assumption"],
    [
        ("EMRGY turbines", "2 turbines at 22 kW each"),
        ("Floating PV", "75 kWdc"),
        ("Canal lining", "500 m2 lined; 8,000 m3/year avoided water loss"),
    ],
)

Physical asset,Key assumption
EMRGY turbines,2 turbines at 22 kW each
Floating PV,75 kWdc
Canal lining,"500 m2 lined; 8,000 m3/year avoided water loss"


## 4. Define Shared Assets

Shared assets depend on the physical asset portfolio. Here, cabling sizes itself from total portfolio capacity.

In [4]:
shared_assets = [
    Cabling(name="Cabling", length_m=120),
]

portfolio = AssetPortfolio(assets=physical_assets + shared_assets)
portfolio_result = portfolio.evaluate(site, resources)

table(
    ["Asset", "Annual energy", "Capacity output", "Year 0 cost", "Annual recurring cost"],
    [
        (
            result.asset_name,
            f"{number(sum(output.quantity for output in result.outputs if output.kind == 'energy_mwh'))} MWh",
            f"{number(sum(output.quantity for output in result.outputs if output.kind in ('capacity_kw', 'cable_capacity_kw')))} kW",
            usd(sum(cost.amount_for_year(0) for cost in result.costs)),
            usd(sum(cost.amount_for_year(1) for cost in result.costs if cost.recurring)),
        )
        for result in portfolio_result.asset_results
    ],
)

Asset,Annual energy,Capacity output,Year 0 cost,Annual recurring cost
EMRGY turbines,282.1 MWh,44.0 kW,"$350,000","$10,000"
Floating PV,134.6 MWh,60.0 kW,"$97,500","$1,650"
Canal lining,0.0 MWh,0.0 kW,"$22,500",$0
Cabling,0.0 MWh,104.0 kW,"$154,760",$774


## 5. Inspect Standardized Outputs

This is the handoff from engineering to commercial valuation: annual MWh, kW, PV area, cable capacity, and avoided water loss in cubic meters.

In [5]:
table(
    ["Source asset", "Output", "Quantity", "Unit", "Period"],
    [
        (
            output.source_asset,
            output.kind,
            number(output.quantity, 2),
            output.unit,
            output.period,
        )
        for output in portfolio_result.outputs
    ],
)

Source asset,Output,Quantity,Unit,Period
EMRGY turbines,energy_mwh,282.13,MWh,annual
EMRGY turbines,capacity_kw,44.00,kW,annual
Floating PV,energy_mwh,134.63,MWh,annual
Floating PV,capacity_kw,60.00,kW,annual
Floating PV,pv_surface_area_m2,525.00,m2,project
Canal lining,avoided_water_loss_m3,"8,000.00",m3,annual
Cabling,cable_capacity_kw,104.00,kW,annual


## 6. Define Revenue Parameters

Revenue models monetize physical outputs. Energy gets a $/MWh value, capacity gets a $/kW-year value, and avoided water loss gets a $/m3 value.

In [6]:
energy_price_per_mwh = 140
capacity_price_per_kw_year = 25
water_value_per_m3 = 0.50
revenue_escalation = 0.02

revenue_models = [
    FixedPriceEnergyRevenue(
        name="Energy revenue",
        price_per_mwh=energy_price_per_mwh,
        escalation_rate=revenue_escalation,
    ),
    CapacityPaymentRevenue(
        name="Capacity value",
        price_per_kw_year=capacity_price_per_kw_year,
    ),
    WaterSavingsRevenue(
        name="Water savings",
        price_per_m3=water_value_per_m3,
        escalation_rate=revenue_escalation,
    ),
]

table(
    ["Revenue parameter", "Value"],
    [
        ("Energy price", f"{usd(energy_price_per_mwh)}/MWh"),
        ("Capacity price", f"{usd(capacity_price_per_kw_year)}/kW-year"),
        ("Water value", f"{usd(water_value_per_m3, 2)}/m3"),
        ("Escalation", f"{revenue_escalation:.1%}"),
    ],
)

Revenue parameter,Value
Energy price,$140/MWh
Capacity price,$25/kW-year
Water value,$0.50/m3
Escalation,2.0%


## 7. Run Revenue And Finance

The finance layer sees only cost lines and revenue lines. It does not need to know which asset produced which physical output.

In [7]:
discount_rate = 0.08
investment_tax_credit = 0.30

engine = ValuationEngine(
    revenue_models=revenue_models,
    finance_model=FinanceModel(
        FinanceAssumptions(
            analysis_years=25,
            discount_rate=discount_rate,
            investment_tax_credit_fraction=investment_tax_credit,
        )
    ),
)

valuation = engine.evaluate(site, resources, portfolio)
metrics = valuation.finance_result.metrics

table(
    ["Revenue line", "Starting annual amount", "Escalation"],
    [
        (line.name, usd(line.amount_usd), f"{line.escalation_rate:.1%}")
        for line in valuation.revenue_lines
    ],
)

table(
    ["Metric", "Value"],
    [
        ("Annual energy", f"{number(valuation.portfolio_result.annual_output('energy_mwh', 'MWh'))} MWh"),
        ("Portfolio capacity", f"{number(valuation.portfolio_result.annual_output('capacity_kw', 'kW'))} kW"),
        ("Avoided water loss", f"{number(valuation.portfolio_result.annual_output('avoided_water_loss_m3', 'm3'))} m3/year"),
        ("Discount rate", f"{discount_rate:.1%}"),
        ("Investment tax credit", f"{investment_tax_credit:.0%}"),
        ("NPV", usd(metrics.npv_usd)),
        ("IRR", percent(metrics.irr)),
        ("LCOE", "n/a" if metrics.lcoe_usd_per_mwh is None else f"{usd(metrics.lcoe_usd_per_mwh)}/MWh"),
        ("Simple payback", "n/a" if metrics.simple_payback_year is None else f"Year {metrics.simple_payback_year}"),
    ],
)

Revenue line,Starting annual amount,Escalation
Energy revenue,"$58,347",2.0%
Capacity value,"$2,600",0.0%
Water savings,"$4,000",2.0%


Metric,Value
Annual energy,416.8 MWh
Portfolio capacity,104.0 kW
Avoided water loss,"8,000.0 m3/year"
Discount rate,8.0%
Investment tax credit,30%
NPV,"$234,101"
IRR,12.8%
LCOE,$131/MWh
Simple payback,Year 8


## 8. Cash Flow View

In [8]:
table(
    ["Year", "Revenue", "Capex", "Opex", "Incentives", "Tax", "Net cash flow"],
    [
        (
            cf.year,
            usd(cf.revenue_usd),
            usd(cf.capex_usd + cf.development_usd),
            usd(cf.opex_usd),
            usd(cf.incentives_usd),
            usd(cf.tax_usd),
            usd(cf.net_cash_flow_usd),
        )
        for cf in valuation.finance_result.cash_flows[:]
    ],
)

Year,Revenue,Capex,Opex,Incentives,Tax,Net cash flow
0,$0,"$624,760",$0,$0,$0,"$-624,760"
1,"$64,947",$0,"$12,424","$187,428",$0,"$239,951"
2,"$66,194",$0,"$12,424",$0,$0,"$53,770"
3,"$67,465",$0,"$12,424",$0,$0,"$55,042"
4,"$68,763",$0,"$12,424",$0,$0,"$56,339"
5,"$70,086",$0,"$12,424",$0,$0,"$57,662"
6,"$71,436",$0,"$12,424",$0,$0,"$59,012"
7,"$72,812",$0,"$12,424",$0,$0,"$60,389"
8,"$74,217",$0,"$12,424",$0,$0,"$61,793"
9,"$75,649",$0,"$12,424",$0,$0,"$63,225"


## Interview Talk Track

- The site and resource profile define the physical context.
- Physical assets convert that context into annual energy, capacity, and water-savings outputs.
- Shared assets, like cabling, scale from the physical portfolio.
- Revenue models assign prices to physical outputs.
- Finance turns costs and revenues into cash flows and NPV.